# NHLA Board Grade Classification

This notebook builds a classification pipeline for NHLA grading from board-face images and XML metadata. It mirrors the repo's notebook structure and adds TensorBoard logging, Optuna tuning, and classification-focused metrics.

Goals:
- process the extracted NHLA CSVs
- split data by board ID
- train a top/bottom side classifier
- optionally train a dual-face board classifier
- evaluate using classification metrics, confusion matrix, and AUC

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path("../").resolve()
CSV_PATH = PROJECT_ROOT / "data" / "26-8-18 Test" / "nhla_dataset" / "nhla_face_labels.csv"
OUTPUT_DIR = PROJECT_ROOT / "models" / "NHLA_Classification"
CSV_PATH.exists()

In [ ]:
from notebooks.NHLA_Classification import (
    train_model,
    load_dataset,
    build_class_mapping,
    split_board_ids,
    evaluate_model,
)

# Example usage: face model
# train_model(
#     csv_path=str(CSV_PATH),
#     output_dir=str(OUTPUT_DIR),
#     mode="face",
#     epochs=12,
#     batch_size=16,
#     learning_rate=1e-3,
#     weight_decay=1e-4,
#     seed=42,
#     val_split=0.2,
# )

## 1. Load and inspect the extracted data

We already generated the CSVs using the NHLA extraction script. This section verifies the labels and classes before training.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(df.head())
print("Unique labels:", sorted(df["grade_label"].unique().tolist()))
print("Rows:", len(df))

## 2. Train the side-only classifier

This model predicts the grade for each face using the image alone. It is the simplest model to deploy because the scanner already combines the top and bottom decisions into the final board grade.

In [ ]:
result_face = train_model(
    csv_path=str(CSV_PATH),
    output_dir=str(OUTPUT_DIR / "face"),
    mode="face",
    epochs=12,
    batch_size=16,
    learning_rate=1e-3,
    weight_decay=1e-4,
    seed=42,
    val_split=0.2,
)
print(result_face["metrics"])

## 3. Optional: board-pair model

This version uses both faces together. It is useful as a second model when you want grader overrides and paired-face context to influence the overall board grade.

In [ ]:
board_csv = PROJECT_ROOT / "data" / "26-8-18 Test" / "nhla_dataset" / "nhla_board_labels.csv"
board_result = train_model(
    csv_path=str(board_csv),
    output_dir=str(OUTPUT_DIR / "board"),
    mode="board",
    epochs=12,
    batch_size=8,
    learning_rate=1e-3,
    weight_decay=1e-4,
    seed=42,
    val_split=0.2,
)
print(board_result["metrics"])

## 4. Metrics to track after training

Classification metrics included here:
- accuracy
- balanced accuracy
- macro precision / recall / F1
- weighted precision / recall / F1
- micro F1
- per-class precision / recall / F1
- confusion matrix
- ROC AUC (one-vs-rest for multiclass)
- classification report

In [ ]:
# Example of reading metrics saved by the script
metrics_path = OUTPUT_DIR / "face" / "metrics.json"
if metrics_path.exists():
    with metrics_path.open("r") as f:
        metrics = json.load(f)
    print(json.dumps({k: metrics[k] for k in [
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "weighted_f1",
        "micro_f1",
        "roc_auc_ovr_macro",
    ]}, indent=2))

## 5. Optional Optuna tuning

Uncomment and run the following if you want hyperparameter search across learning rate, batch size, and weight decay.

In [ ]:
from notebooks.NHLA_Classification import run_optuna_tuning

study, best_params = run_optuna_tuning(
    csv_path=str(CSV_PATH),
    mode="face",
    output_root=str(OUTPUT_DIR / "optuna"),
    n_trials=10,
    epochs=8,
    seed=42,
    val_split=0.2,
)

print("Best study value:", study.best_value)
print("Best params:", best_params)
